# AVV Classification: WasteItem → AVVCode (3-Step Hierarchical)

Classifies each WasteItem in 3 small LLM calls instead of 1 large one:
1. **Chapter** — pick from 20 AVV chapters (~272 tokens)
2. **Subgroup** — pick from ~5-30 subgroups within that chapter
3. **Specific code** — pick from ~5-15 codes within that subgroup

Each call uses ~300-500 tokens total. Works with small models (8B). No rate limit issues.

**Steps:**
1. Import AVV codes into Neo4j (structured ETL, no LLM)
2. Load AVV hierarchy for dynamic context filtering
3. Query all WasteItem nodes
4. Classify each item through 3 hierarchical steps
5. Write `CLASSIFIED_AS` relationships to Neo4j

**Prerequisites:**
- Neo4j running with WasteItem nodes already imported
- `.env` configured with `LLM_API_KEY`

In [ ]:
import sys
sys.path.insert(0, "..")

import logging
from collections import Counter

from src.db import neo4j_db
from src.llm_etl.extraction_chain import ExtractionChain
from src.etl.avv_codes import parse_avv_csv, CHAPTER_NAMES, DATA_FILE
from src.etl.avv_classification import (
    ChapterClassification, SubgroupClassification, AVVClassification,
    build_avv_hierarchy, classify_single_item,
)

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

## 1. Import AVV Codes (Structured ETL)

First, populate the graph with AVVCode nodes from the CSV catalog.
This is a one-time structured import (no LLM needed).

In [ ]:
from src.etl.avv_codes import import_avv_codes

stats = import_avv_codes(dry_run=False)

print(f"AVV codes imported:")
print(f"  Chapters (L1):   {stats['level1']}")
print(f"  Subgroups (L2):  {stats['level2']}")
print(f"  Types (L3):      {stats['level3']}")
print(f"  Total:           {stats['total']}")

## 2. Load AVV Hierarchy

Build the 3-level hierarchy for dynamic context filtering.
Each classification step only sees the relevant subset of codes.

In [ ]:
avv = build_avv_hierarchy()

print(f"AVV hierarchy loaded:")
print(f"  Chapters (L1): {len(avv['level1'])}")
print(f"  Subgroups (L2): {len(avv['level2'])}")
print(f"  Codes (L3): {len(avv['level3'])}")
print(f"\nStep 1 context ({len(avv['chapter_lines'])} chars, ~{len(avv['chapter_lines'])//4} tokens):")
print(avv["chapter_lines"])

## 3. Query WasteItem Nodes

Get all WasteItem names from Neo4j. These are what we want to classify.

In [ ]:
waste_items = neo4j_db.query(
    "MATCH (w:WasteItem) RETURN w.name AS name ORDER BY w.name"
)
item_names = [r["name"] for r in waste_items if r["name"]]

print(f"Found {len(item_names)} WasteItem nodes")
print(f"Sample: {item_names[:10]}")

## 4. Initialize ExtractionChain

Using `ExtractionChain.extract_single()` directly (not Pipeline) because
we want single entities per call and handle DB writing ourselves.

In [ ]:
chain = ExtractionChain(temperature=0.0, structured_output_method="json_mode")

print("ExtractionChain ready (json_mode, temperature=0.0)")
print(f"Will classify {len(item_names)} items × 3 steps = {len(item_names) * 3} LLM calls")
print(f"Estimated tokens per call: ~300-500")

## 5. Test: Classify a Single Item

Run the 3-step classification on one item to verify each step produces valid codes.

In [ ]:
test_name = item_names[0] if item_names else "Altöl"
print(f"Test item: {test_name}\n")

result = classify_single_item(test_name, avv, chain, verbose=True)

if result:
    hz = "*" if result.is_hazardous else ""
    print(f"\nResult: {result.avv_code}{hz}")
    print(f"Hazardous: {result.is_hazardous}")
    print(f"Reasoning: {result.reasoning}")
else:
    print("Classification failed!")

## 6. Run Batch Classification

Classify multiple items with delay between each (3 calls per item).

In [ ]:
import time

LIMIT = 10  # Set to 0 for all items
DELAY = 2.0  # Seconds between items

names_to_classify = item_names[:LIMIT] if LIMIT > 0 else item_names
all_classifications: list[tuple[str, AVVClassification]] = []
error_list: list[str] = []

for i, name in enumerate(names_to_classify, 1):
    print(f"[{i}/{len(names_to_classify)}] {name}", end=" ... ")

    try:
        result = classify_single_item(name, avv, chain, verbose=False)
        if result:
            all_classifications.append((name, result))
            hz = "*" if result.is_hazardous else ""
            print(f"-> {result.avv_code}{hz}")
        else:
            error_list.append(name)
            print("FAILED")
    except Exception as e:
        error_list.append(name)
        print(f"ERROR: {e}")

    if DELAY > 0 and i < len(names_to_classify):
        time.sleep(DELAY)

print(f"\nClassified: {len(all_classifications)}/{len(names_to_classify)}")
print(f"Failed: {len(error_list)}")

## 7. Inspect Results

Review the classifications before writing to the database.

In [ ]:
# Summary by chapter
chapter_counts = Counter(c.avv_code[:2] for _, c in all_classifications)
print("Classifications by AVV chapter:")
for chapter, count in chapter_counts.most_common():
    name = CHAPTER_NAMES.get(chapter, "?")
    print(f"  {chapter} - {name}: {count}")

hazardous_count = sum(1 for _, c in all_classifications if c.is_hazardous)
print(f"\nHazardous items: {hazardous_count}")
print(f"Non-hazardous: {len(all_classifications) - hazardous_count}")

# Show classifications
print("\nAll classifications:")
for name, c in all_classifications:
    hz = "*" if c.is_hazardous else " "
    print(f"  {name:30s} -> {c.avv_code}{hz}  ({c.reasoning[:60]})")

if error_list:
    print(f"\nFailed items ({len(error_list)}):")
    for name in error_list:
        print(f"  - {name}")

In [ ]:
# Validate code hierarchy consistency: step3 starts with step2 starts with step1
print("Code hierarchy validation:")
for name, c in all_classifications:
    code = c.avv_code.zfill(6)
    chapter = code[:2]
    subgroup = code[:4]
    l3_entry = avv["level3"].get(code)
    if l3_entry:
        parent_sg = l3_entry["parent_code"]
        sg_entry = avv["level2"].get(parent_sg, {})
        parent_ch = sg_entry.get("parent_code", "")
        ok = parent_sg == subgroup and parent_ch == chapter
        status = "OK" if ok else "MISMATCH"
        print(f"  {name:30s} {code} ch={chapter} sg={subgroup} -> {status}")
    else:
        print(f"  {name:30s} {code} -> NOT IN AVV DATA")

## 8. Write CLASSIFIED_AS Relationships to Neo4j

For each classification, create a `(WasteItem)-[:CLASSIFIED_AS]->(AVVCode)` relationship.
Uses MERGE for idempotency.

In [ ]:
DRY_RUN = True  # Set to False to actually write

created = 0
skipped = 0

if not DRY_RUN:
    with neo4j_db.session() as session:
        for name, c in all_classifications:
            code = c.avv_code.zfill(6)
            result = session.run(
                """
                MATCH (w:WasteItem {name: $name})
                MATCH (a:AVVCode {code: $code})
                MERGE (w)-[r:CLASSIFIED_AS]->(a)
                ON CREATE SET
                    r.reasoning = $reasoning,
                    r.is_hazardous = $is_hazardous,
                    r.classified_by = 'llm_hierarchical',
                    r.created_at = datetime()
                RETURN w.name AS name, a.code AS code
                """,
                {
                    "name": name,
                    "code": code,
                    "reasoning": c.reasoning,
                    "is_hazardous": c.is_hazardous,
                },
            )
            record = result.single()
            if record:
                created += 1
            else:
                skipped += 1

    print(f"CLASSIFIED_AS relationships created: {created}")
    print(f"Skipped (name/code mismatch): {skipped}")
else:
    print(f"DRY RUN: Would create {len(all_classifications)} CLASSIFIED_AS relationships")
    print("Set DRY_RUN = False above and re-run to write.")

## 9. Verify in Neo4j

Run validation queries to confirm the classification results.

In [ ]:
# Count classifications by chapter
results = neo4j_db.query("""
    MATCH (w:WasteItem)-[:CLASSIFIED_AS]->(a:AVVCode)
    WITH a.code AS code, substring(a.code, 0, 2) AS chapter, count(w) AS items
    RETURN chapter, sum(items) AS total_items
    ORDER BY total_items DESC
""")

print("Classifications by AVV chapter:")
for r in results:
    print(f"  Chapter {r['chapter']}: {r['total_items']} items")

# Unclassified items
unclassified = neo4j_db.query("""
    MATCH (w:WasteItem)
    WHERE NOT (w)-[:CLASSIFIED_AS]->()
    RETURN count(w) AS count
""")
print(f"\nUnclassified WasteItems: {unclassified[0]['count']}")

# Hazardous items
hazardous = neo4j_db.query("""
    MATCH (w:WasteItem)-[r:CLASSIFIED_AS]->(a:AVVCode)
    WHERE a.is_hazardous = true
    RETURN w.name AS name, a.code AS code
    ORDER BY w.name
    LIMIT 20
""")
print(f"\nHazardous waste items (first 20):")
for r in hazardous:
    print(f"  {r['name']} -> {r['code']}")